# In-depth Evaluation

This notebook is used to generate plots and result tables in order to test the quality of the It complements `06_test_aggregation.ipynb`, which checks project-wide tests and deployment artifacts.

The main experiments are:

1. a corrected nested Leave-One-Subject-Out (LOSO) protocol, where the outer test subject is never used for scaling, early stopping, or learning-rate decisions;
2. a comparison between Logistic Regression, Random Forest, MLP, 1D-CNN, and the deployed LSTM under the same subject splits;
3. out-of-fold confusion matrices, threshold sensitivity, probability calibration, ROC, and precision-recall analysis;
4. subject-independent sensor-group ablation;
5. simulated personalization using a small amount of held-out-subject feedback;
6. a controlled sample-weighted FedAvg comparison against centralized training.

`full` mode runs all 15 outer folds and writes thesis-ready artifacts under `ml/evaluation`. `smoke` mode uses three subjects and reduced epochs, writing into `ml/evaluation/smoke`; smoke results must not be used in the thesis.


In [1]:
import gc
import json
import os
import pickle
import platform
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import Markdown, display
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight


def locate_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "ml" / "src").exists() and (candidate / "server").exists():
            return candidate
    raise RuntimeError(f"Cannot locate the MindWave repository from {current}")


REPO_ROOT = locate_repo_root()
ML_DIR = REPO_ROOT / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from src.config import SEED, set_global_seed
from src.model import build_lstm

RUN_MODE = os.environ.get("MINDWAVE_EVAL_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("MINDWAVE_EVAL_MODE must be 'full' or 'smoke'")

FULL_RUN = RUN_MODE == "full"
DATA_PATH = ML_DIR / "data" / "processed" / "wesad_wrist.npz"
EVAL_ROOT = ML_DIR / "evaluation" if FULL_RUN else ML_DIR / "evaluation" / "smoke"
RESULTS_DIR = EVAL_ROOT / "results"
PLOTS_DIR = EVAL_ROOT / "plots"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ["Logistic Regression", "Random Forest", "MLP", "1D-CNN", "LSTM"]
DEEP_EPOCHS = 40 if FULL_RUN else 2
EARLY_STOPPING_PATIENCE = 8 if FULL_RUN else 1
INNER_VALIDATION_SUBJECTS = 2 if FULL_RUN else 1
RF_TREES = 300 if FULL_RUN else 25
PERSONALIZATION_BUDGETS = (4, 8, 16) if FULL_RUN else (4,)
PERSONALIZATION_EPOCHS = 2 if FULL_RUN else 1
FL_ROUNDS = 3 if FULL_RUN else 1
FL_LOCAL_EPOCHS = 1
PRODUCTION_THRESHOLD = 0.85
N_CALIBRATION_BINS = 10

set_global_seed(SEED)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "font.size": 10})

generated = []


def save_table(df: pd.DataFrame, name: str, description: str, index: bool = False) -> Path:
    path = RESULTS_DIR / name
    df.to_csv(path, index=index)
    generated.append({"artifact": str(path.relative_to(REPO_ROOT)), "kind": "table", "description": description})
    print(f"  -> wrote {path.relative_to(REPO_ROOT)} ({len(df)} rows)")
    return path


def save_figure(fig, name: str, description: str) -> Path:
    path = PLOTS_DIR / name
    fig.savefig(path, dpi=180)
    plt.close(fig)
    generated.append({"artifact": str(path.relative_to(REPO_ROOT)), "kind": "plot", "description": description})
    print(f"  -> wrote {path.relative_to(REPO_ROOT)}")
    return path


print(f"Repository: {REPO_ROOT}")
print(f"Run mode:   {RUN_MODE}")
print(f"Python:     {sys.version.split()[0]}")
print(f"TensorFlow: {tf.__version__}")
print(f"Outputs:    {EVAL_ROOT.relative_to(REPO_ROOT)}")
if not FULL_RUN:
    display(Markdown("**Smoke mode is active. Its results are for code verification only.**"))


Repository: C:\Users\daniion6\AndroidStudioProjects\MindWave
Run mode:   full
Python:     3.12.10
TensorFlow: 2.21.0
Outputs:    ml\evaluation


## Dataset and Evaluation Environment

The class distribution and subject composition are saved before training. The environment table makes it possible to distinguish these desktop research experiments from measurements collected later on a physical Android phone and Wear OS watch.


In [2]:
data = np.load(DATA_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
subject_ids = data["subject_ids"].astype(str)
feature_names = [str(name) for name in data["feature_names"]]


def subject_key(subject: str) -> int:
    value = str(subject)
    return int(value[1:]) if value.startswith("S") and value[1:].isdigit() else 999


all_subjects = sorted(np.unique(subject_ids), key=subject_key)
outer_subjects = all_subjects if FULL_RUN else all_subjects[:3]
selected_mask = np.isin(subject_ids, outer_subjects)
X_eval, y_eval, subjects_eval = X[selected_mask], y[selected_mask], subject_ids[selected_mask]

balance = (
    pd.DataFrame({"subject": subjects_eval, "label": y_eval})
    .assign(class_name=lambda frame: frame["label"].map({0: "normal", 1: "stress"}))
    .groupby(["subject", "class_name"])
    .size()
    .unstack(fill_value=0)
    .reindex(outer_subjects)
)
for required in ("normal", "stress"):
    if required not in balance.columns:
        balance[required] = 0
balance = balance[["normal", "stress"]].astype(int)
balance["total"] = balance.sum(axis=1)
balance["stress_share"] = balance["stress"] / balance["total"]
save_table(balance.reset_index(), "indepth_class_balance_by_subject.csv", "Class balance for subjects included in the in-depth evaluation")

environment = pd.DataFrame(
    [
        ("run_mode", RUN_MODE),
        ("platform", platform.platform()),
        ("python", sys.version.split()[0]),
        ("tensorflow", tf.__version__),
        ("numpy", np.__version__),
        ("subjects", ", ".join(outer_subjects)),
        ("windows", len(X_eval)),
        ("input_shape", str(tuple(X_eval.shape[1:]))),
        ("evaluation_note", "Desktop research evaluation; not physical-device profiling"),
    ],
    columns=["property", "value"],
)
save_table(environment, "indepth_environment.csv", "Software, dataset, and execution environment")

fig, ax = plt.subplots(figsize=(8.5, 4.4))
positions = np.arange(len(balance))
ax.bar(positions, balance["normal"], color="#2a9d8f", label="normal")
ax.bar(positions, balance["stress"], bottom=balance["normal"], color="#e76f51", label="stress")
ax.set_xticks(positions)
ax.set_xticklabels(balance.index)
ax.set_ylabel("Windows")
ax.set_xlabel("WESAD subject")
ax.set_title("Class balance by subject after window filtering")
ax.legend()
fig.tight_layout()
save_figure(fig, "indepth_class_balance_by_subject.png", "Class balance by evaluated WESAD subject")
balance


  -> wrote ml\evaluation\results\indepth_class_balance_by_subject.csv (15 rows)
  -> wrote ml\evaluation\results\indepth_environment.csv (9 rows)
  -> wrote ml\evaluation\plots\indepth_class_balance_by_subject.png


class_name,normal,stress,total,stress_share
subject,,,,
S2,73,37,110,0.336364
S3,72,39,111,0.351351
S4,74,39,113,0.345133
S5,76,40,116,0.344828
S6,75,40,115,0.347826
S7,76,39,115,0.339130
S8,74,41,115,0.356522
S9,76,40,116,0.344828
S10,76,45,121,0.371901


## Shared Metrics and Subject-independent Split

Each outer fold reserves one complete subject for testing. One or two subjects from the remaining set are selected for validation using a deterministic group split. The scaler and noisy augmentation are fitted only on the inner training subjects. Therefore, neither feature statistics nor early-stopping decisions use the outer test subject.


In [3]:
def fit_scaler_3d(values: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(values.reshape(-1, values.shape[-1]))
    return scaler


def apply_scaler_3d(scaler: StandardScaler, values: np.ndarray) -> np.ndarray:
    shape = values.shape
    return scaler.transform(values.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)


def expected_calibration_error(y_true: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:
    edges = np.linspace(0.0, 1.0, bins + 1)
    total = len(y_true)
    value = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        include = (probabilities >= lower) & (probabilities < upper if upper < 1.0 else probabilities <= upper)
        if not np.any(include):
            continue
        confidence = float(probabilities[include].mean())
        observed = float(y_true[include].mean())
        value += include.sum() / total * abs(confidence - observed)
    return float(value)


def metric_row(y_true: np.ndarray, probabilities: np.ndarray, threshold: float = 0.5) -> dict:
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= threshold).astype(int)
    precision, recall, stress_f1, _ = precision_recall_fscore_support(
        y_true, predictions, average="binary", pos_label=1, zero_division=0
    )
    return {
        "n_windows": int(len(y_true)),
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "macro_f1": float(f1_score(y_true, predictions, average="macro", zero_division=0)),
        "stress_precision": float(precision),
        "stress_recall": float(recall),
        "stress_f1": float(stress_f1),
        "roc_auc": float(roc_auc_score(y_true, probabilities)) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(y_true, probabilities)) if len(np.unique(y_true)) == 2 else np.nan,
        "brier_score": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(log_loss(y_true, np.column_stack([1.0 - probabilities, probabilities]), labels=[0, 1])),
        "ece": expected_calibration_error(y_true, probabilities, N_CALIBRATION_BINS),
    }


def inner_subject_split(available_subjects: list[str], fold_index: int) -> tuple[list[str], list[str]]:
    groups = np.asarray(sorted(available_subjects, key=subject_key))
    validation_count = min(INNER_VALIDATION_SUBJECTS, max(1, len(groups) - 1))
    splitter = GroupShuffleSplit(n_splits=1, test_size=validation_count, random_state=SEED + fold_index)
    dummy = np.zeros(len(groups))
    train_indices, validation_indices = next(splitter.split(dummy, groups=groups))
    return groups[train_indices].tolist(), groups[validation_indices].tolist()


def augment_scaled_windows(values: np.ndarray, labels: np.ndarray, seed: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, 0.05, size=values.shape).astype(np.float32)
    return np.concatenate([values, values + noise]), np.concatenate([labels, labels])


def class_weights(labels: np.ndarray) -> dict[int, float]:
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=labels)
    return {int(label): float(weight) for label, weight in zip(classes, weights)}


## Comparable Model Families

The classical and neural alternatives are evaluated with the same outer test subjects, inner validation subjects, normalized inputs, and training augmentation. Desktop fit and prediction timings are comparative development measurements only; physical-device profiling is handled separately.


In [4]:
def build_mlp(input_shape: tuple[int, int]) -> tf.keras.Model:
    inputs = tf.keras.layers.Input(shape=input_shape)
    hidden = tf.keras.layers.Flatten()(inputs)
    hidden = tf.keras.layers.Dense(64, activation="relu")(hidden)
    hidden = tf.keras.layers.Dropout(0.3)(hidden)
    hidden = tf.keras.layers.Dense(32, activation="relu")(hidden)
    outputs = tf.keras.layers.Dense(2, activation="softmax")(hidden)
    model = tf.keras.Model(inputs, outputs, name="mindwave_mlp_baseline")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def build_cnn(input_shape: tuple[int, int]) -> tf.keras.Model:
    inputs = tf.keras.layers.Input(shape=input_shape)
    hidden = tf.keras.layers.Conv1D(32, 3, padding="same", activation="relu")(inputs)
    hidden = tf.keras.layers.Conv1D(32, 3, padding="same", activation="relu")(hidden)
    hidden = tf.keras.layers.GlobalAveragePooling1D()(hidden)
    hidden = tf.keras.layers.Dense(32, activation="relu")(hidden)
    outputs = tf.keras.layers.Dense(2, activation="softmax")(hidden)
    model = tf.keras.Model(inputs, outputs, name="mindwave_1dcnn_baseline")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def one_window_latency_ms(model_name: str, model, sample: np.ndarray, repetitions: int = 20) -> float:
    repetitions = min(repetitions, 5) if not FULL_RUN else repetitions
    if model_name in {"Logistic Regression", "Random Forest"}:
        prepared = sample.reshape(1, -1)
        model.predict_proba(prepared)
        start = time.perf_counter()
        for _ in range(repetitions):
            model.predict_proba(prepared)
    else:
        model(sample[:1], training=False).numpy()
        start = time.perf_counter()
        for _ in range(repetitions):
            model(sample[:1], training=False).numpy()
    return (time.perf_counter() - start) * 1000.0 / repetitions


def fit_candidate(
    model_name: str,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_validation: np.ndarray,
    y_validation: np.ndarray,
    X_test: np.ndarray,
    fold_seed: int,
):
    started = time.perf_counter()
    if model_name == "Logistic Regression":
        model = LogisticRegression(max_iter=2500, class_weight="balanced", random_state=fold_seed)
        model.fit(X_train.reshape(len(X_train), -1), y_train)
        probabilities = model.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]
        parameter_count = int(model.coef_.size + model.intercept_.size)
        artifact_bytes = len(pickle.dumps(model))
    elif model_name == "Random Forest":
        model = RandomForestClassifier(
            n_estimators=RF_TREES,
            class_weight="balanced_subsample",
            random_state=fold_seed,
            n_jobs=1,
        )
        model.fit(X_train.reshape(len(X_train), -1), y_train)
        probabilities = model.predict_proba(X_test.reshape(len(X_test), -1))[:, 1]
        parameter_count = int(sum(estimator.tree_.node_count for estimator in model.estimators_))
        artifact_bytes = len(pickle.dumps(model))
    else:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(fold_seed)
        if model_name == "MLP":
            model = build_mlp(X_train.shape[1:])
        elif model_name == "1D-CNN":
            model = build_cnn(X_train.shape[1:])
        elif model_name == "LSTM":
            model = build_lstm(input_shape=X_train.shape[1:], n_classes=2)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=EARLY_STOPPING_PATIENCE,
                restore_best_weights=True,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                patience=max(1, EARLY_STOPPING_PATIENCE // 2),
                factor=0.5,
                min_lr=1e-5,
            ),
        ]
        model.fit(
            X_train,
            y_train,
            validation_data=(X_validation, y_validation),
            epochs=DEEP_EPOCHS,
            batch_size=64,
            class_weight=class_weights(y_train),
            callbacks=callbacks,
            verbose=0,
        )
        probabilities = model.predict(X_test, verbose=0)[:, 1]
        parameter_count = int(model.count_params())
        artifact_bytes = int(sum(weight.nbytes for weight in model.get_weights()))

    fit_seconds = time.perf_counter() - started
    latency_ms = one_window_latency_ms(model_name, model, X_test[:1])
    return model, probabilities.astype(float), fit_seconds, latency_ms, parameter_count, artifact_bytes


## Corrected Nested LOSO and Out-of-fold Predictions

This is the longest notebook section. In `full` mode it trains every model for each of the 15 outer subjects. The saved out-of-fold probabilities are the only probabilities used later for confusion matrices, threshold sensitivity, and calibration.


In [5]:
fold_metrics = []
oof_predictions = []
ablation_predictions = []
personalization_rows = []
split_rows = []

FEATURE_GROUPS = {
    "HRV/BVP": np.arange(0, 7),
    "EDA": np.arange(7, 14),
    "Temperature": np.arange(14, 19),
    "Movement": np.arange(19, 23),
}
PURGE_WINDOWS = 3  # 60-second windows with a 15-second stride overlap for three neighboring starts.


def personalization_indices(labels: np.ndarray, total_budget: int) -> tuple[np.ndarray, np.ndarray]:
    per_class = max(1, total_budget // 2)
    adaptation = []
    for label in (0, 1):
        candidates = np.where(labels == label)[0]
        if len(candidates) < per_class + 1:
            return np.array([], dtype=int), np.array([], dtype=int)
        adaptation.extend(candidates[:per_class].tolist())
    adaptation = np.asarray(sorted(adaptation), dtype=int)
    allowed = np.ones(len(labels), dtype=bool)
    for index in adaptation:
        allowed[max(0, index - PURGE_WINDOWS): min(len(labels), index + PURGE_WINDOWS + 1)] = False
    evaluation = np.where(allowed)[0]
    if len(np.unique(labels[evaluation])) < 2:
        return np.array([], dtype=int), np.array([], dtype=int)
    return adaptation, evaluation


for fold_index, test_subject in enumerate(outer_subjects):
    available = [subject for subject in outer_subjects if subject != test_subject]
    inner_train_subjects, validation_subjects = inner_subject_split(available, fold_index)
    train_mask = np.isin(subjects_eval, inner_train_subjects)
    validation_mask = np.isin(subjects_eval, validation_subjects)
    test_mask = subjects_eval == test_subject

    scaler = fit_scaler_3d(X_eval[train_mask])
    X_train = apply_scaler_3d(scaler, X_eval[train_mask])
    X_validation = apply_scaler_3d(scaler, X_eval[validation_mask])
    X_test = apply_scaler_3d(scaler, X_eval[test_mask])
    y_train, y_validation, y_test = y_eval[train_mask], y_eval[validation_mask], y_eval[test_mask]
    X_train_aug, y_train_aug = augment_scaled_windows(X_train, y_train, SEED + fold_index)

    split_rows.append(
        {
            "outer_test_subject": test_subject,
            "inner_train_subjects": " ".join(inner_train_subjects),
            "validation_subjects": " ".join(validation_subjects),
            "n_train": len(X_train),
            "n_validation": len(X_validation),
            "n_test": len(X_test),
        }
    )
    print(f"[{fold_index + 1}/{len(outer_subjects)}] test={test_subject}; validation={validation_subjects}")

    for model_index, model_name in enumerate(MODEL_NAMES):
        model, probabilities, fit_seconds, latency_ms, parameter_count, artifact_bytes = fit_candidate(
            model_name,
            X_train_aug,
            y_train_aug,
            X_validation,
            y_validation,
            X_test,
            SEED + fold_index * 10 + model_index,
        )
        metrics = metric_row(y_test, probabilities, threshold=0.5)
        fold_metrics.append(
            {
                "subject": test_subject,
                "model": model_name,
                **metrics,
                "fit_seconds": fit_seconds,
                "desktop_latency_ms": latency_ms,
                "parameter_count": parameter_count,
                "artifact_bytes": artifact_bytes,
            }
        )
        original_indices = np.where(selected_mask)[0][test_mask]
        for original_index, true_label, probability in zip(original_indices, y_test, probabilities):
            oof_predictions.append(
                {
                    "sample_index": int(original_index),
                    "subject": test_subject,
                    "model": model_name,
                    "y_true": int(true_label),
                    "probability_stress": float(probability),
                }
            )

        if model_name == "LSTM":
            for modality, columns in {"None": np.array([], dtype=int), **FEATURE_GROUPS}.items():
                ablated = X_test.copy()
                if len(columns):
                    ablated[:, :, columns] = 0.0  # Mean masking in normalized feature space.
                ablated_probabilities = model.predict(ablated, verbose=0)[:, 1]
                for true_label, probability in zip(y_test, ablated_probabilities):
                    ablation_predictions.append(
                        {
                            "subject": test_subject,
                            "masked_modality": modality,
                            "y_true": int(true_label),
                            "probability_stress": float(probability),
                        }
                    )

            base_weights = model.get_weights()
            for feedback_budget in PERSONALIZATION_BUDGETS:
                adaptation_indices, evaluation_indices = personalization_indices(y_test, feedback_budget)
                if not len(adaptation_indices):
                    continue
                before = model.predict(X_test[evaluation_indices], verbose=0)[:, 1]
                personalized = tf.keras.models.clone_model(model)
                personalized.set_weights(base_weights)
                personalized.compile(
                    optimizer=tf.keras.optimizers.SGD(learning_rate=1e-3),
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"],
                )
                personalized.fit(
                    X_test[adaptation_indices],
                    y_test[adaptation_indices],
                    epochs=PERSONALIZATION_EPOCHS,
                    batch_size=min(8, len(adaptation_indices)),
                    verbose=0,
                )
                after = personalized.predict(X_test[evaluation_indices], verbose=0)[:, 1]
                before_metrics = metric_row(y_test[evaluation_indices], before, threshold=0.5)
                after_metrics = metric_row(y_test[evaluation_indices], after, threshold=0.5)
                personalization_rows.append(
                    {
                        "subject": test_subject,
                        "feedback_budget": int(len(adaptation_indices)),
                        "evaluation_windows": int(len(evaluation_indices)),
                        "accuracy_before": before_metrics["accuracy"],
                        "accuracy_after": after_metrics["accuracy"],
                        "accuracy_delta": after_metrics["accuracy"] - before_metrics["accuracy"],
                        "macro_f1_before": before_metrics["macro_f1"],
                        "macro_f1_after": after_metrics["macro_f1"],
                        "macro_f1_delta": after_metrics["macro_f1"] - before_metrics["macro_f1"],
                        "brier_before": before_metrics["brier_score"],
                        "brier_after": after_metrics["brier_score"],
                    }
                )
                del personalized
        del model
        gc.collect()

fold_metrics_df = pd.DataFrame(fold_metrics)
oof_df = pd.DataFrame(oof_predictions)
splits_df = pd.DataFrame(split_rows)
ablation_oof_df = pd.DataFrame(ablation_predictions)
personalization_df = pd.DataFrame(personalization_rows)

save_table(splits_df, "nested_loso_subject_splits.csv", "Outer test and inner validation subjects for each corrected LOSO fold")
save_table(fold_metrics_df, "nested_loso_fold_metrics.csv", "Per-subject metrics for every model under corrected nested LOSO")
save_table(oof_df, "nested_loso_oof_predictions.csv", "Out-of-fold probabilities from untouched outer test subjects")
fold_metrics_df.head()


[1/15] test=S2; validation=['S5', 'S7']

[2/15] test=S3; validation=['S2', 'S5']
[3/15] test=S4; validation=['S3', 'S16']
[4/15] test=S5; validation=['S3', 'S4']
[5/15] test=S6; validation=['S11', 'S17']
[6/15] test=S7; validation=['S15', 'S16']
[7/15] test=S8; validation=['S6', 'S16']
[8/15] test=S9; validation=['S10', 'S14']
[9/15] test=S10; validation=['S6', 'S8']
[10/15] test=S11; validation=['S4', 'S15']
[11/15] test=S13; validation=['S4', 'S11']
[12/15] test=S14; validation=['S3', 'S15']
[13/15] test=S15; validation=['S8', 'S16']
[14/15] test=S16; validation=['S8', 'S15']
[15/15] test=S17; validation=['S4', 'S7']
  -> wrote ml\evaluation\results\nested_loso_subject_splits.csv (15 rows)
  -> wrote ml\evaluation\results\nested_loso_fold_metrics.csv (75 rows)
  -> wrote ml\evaluation\results\nested_loso_oof_predictions.csv (8690 rows)


,subject,model,n_windows,threshold,accuracy,balanced_accuracy,macro_f1,stress_precision,stress_recall,stress_f1,roc_auc,pr_auc,brier_score,log_loss,ece,fit_seconds,desktop_latency_ms,parameter_count,artifact_bytes
0,S2,Logistic Regression,110,0.5,0.900000,0.904665,0.891373,0.809524,0.918919,0.860759,0.976675,0.957655,0.074866,0.308340,0.089621,0.072872,0.104645,277,2929
1,S2,Random Forest,110,0.5,0.990909,0.986486,0.989749,1.000000,0.972973,0.986301,1.000000,1.000000,0.008759,0.050668,0.044364,10.568560,39.344400,52164,4266795
2,S2,MLP,110,0.5,0.981818,0.972973,0.979354,1.000000,0.945946,0.972222,0.998519,0.997243,0.018434,0.079795,0.039176,9.056953,9.351230,19874,79496
3,S2,1D-CNN,110,0.5,0.963636,0.945946,0.958095,1.000000,0.891892,0.942857,0.998889,0.997884,0.030862,0.139872,0.051429,6.430431,6.735965,6466,25864
4,S2,LSTM,110,0.5,0.981818,0.972973,0.979354,1.000000,0.945946,0.972222,0.998149,0.996782,0.016193,0.070112,0.019983,17.062460,92.792640,36066,144264


## Model Comparison and Subject Variation

The aggregate table is computed from concatenated out-of-fold predictions. Mean and standard deviation across subjects are included so that strong average performance cannot hide difficult users.


In [6]:
summary_rows = []
for model_name in MODEL_NAMES:
    model_oof = oof_df[oof_df["model"] == model_name]
    aggregate = metric_row(model_oof["y_true"].to_numpy(), model_oof["probability_stress"].to_numpy(), threshold=0.5)
    per_subject = fold_metrics_df[fold_metrics_df["model"] == model_name]
    summary_rows.append(
        {
            "model": model_name,
            **aggregate,
            "subject_accuracy_mean": per_subject["accuracy"].mean(),
            "subject_accuracy_std": per_subject["accuracy"].std(ddof=0),
            "subject_macro_f1_mean": per_subject["macro_f1"].mean(),
            "subject_macro_f1_std": per_subject["macro_f1"].std(ddof=0),
            "fit_seconds_total": per_subject["fit_seconds"].sum(),
            "desktop_latency_ms_median": per_subject["desktop_latency_ms"].median(),
            "artifact_kb_median": per_subject["artifact_bytes"].median() / 1024.0,
            "parameter_count_median": per_subject["parameter_count"].median(),
        }
    )

model_summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
save_table(model_summary_df, "nested_loso_model_comparison.csv", "Comparable model-family results from concatenated outer-fold predictions")

plot_data = fold_metrics_df.copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for axis, metric, title in zip(axes, ["accuracy", "macro_f1"], ["Accuracy", "Macro-F1"]):
    sns.barplot(data=plot_data, x="model", y=metric, errorbar="sd", color="#2a9d8f", ax=axis)
    axis.set_ylim(0, 1)
    axis.set_title(f"Subject-independent {title}")
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=25)
fig.tight_layout()
save_figure(fig, "nested_loso_model_comparison.png", "Mean and subject-level variation for each evaluated model family")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for model_name in model_summary_df["model"]:
    model_oof = oof_df[oof_df["model"] == model_name]
    y_model = model_oof["y_true"].to_numpy()
    p_model = model_oof["probability_stress"].to_numpy()
    fpr, tpr, _ = roc_curve(y_model, p_model)
    precision, recall, _ = precision_recall_curve(y_model, p_model)
    axes[0].plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={roc_auc_score(y_model, p_model):.3f})")
    axes[1].plot(recall, precision, linewidth=2, label=f"{model_name} (AP={average_precision_score(y_model, p_model):.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="#6c757d", linewidth=1)
axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Outer-fold ROC curves")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Outer-fold precision-recall curves")
for axis in axes:
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1.02)
    axis.legend(fontsize=8, loc="lower left")
fig.tight_layout()
save_figure(fig, "nested_loso_model_roc_pr_comparison.png", "ROC and precision-recall curves from concatenated nested-LOSO predictions")

lstm_subjects = fold_metrics_df[fold_metrics_df["model"] == "LSTM"].copy()
fig, ax = plt.subplots(figsize=(9, 4.2))
lstm_subjects.set_index("subject")[["accuracy", "macro_f1"]].plot(kind="bar", ax=ax, color=["#2a9d8f", "#457b9d"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Corrected LSTM LOSO performance by held-out subject")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
save_figure(fig, "nested_loso_lstm_by_subject.png", "Corrected LSTM accuracy and macro-F1 by outer test subject")
model_summary_df


  -> wrote ml\evaluation\results\nested_loso_model_comparison.csv (5 rows)
  -> wrote ml\evaluation\plots\nested_loso_model_comparison.png
  -> wrote ml\evaluation\plots\nested_loso_lstm_by_subject.png


,model,n_windows,threshold,accuracy,balanced_accuracy,macro_f1,stress_precision,stress_recall,stress_f1,roc_auc,...,log_loss,ece,subject_accuracy_mean,subject_accuracy_std,subject_macro_f1_mean,subject_macro_f1_std,fit_seconds_total,desktop_latency_ms_median,artifact_kb_median,parameter_count_median
2,MLP,1738,0.5,0.941887,0.940661,0.936936,0.902669,0.936482,0.919265,0.984804,...,0.180648,0.026733,0.942354,0.049591,0.937826,0.051119,104.303959,5.437935,77.632812,19874.0
4,LSTM,1738,0.5,0.937860,0.934222,0.932301,0.904153,0.921824,0.912903,0.973173,...,0.219640,0.035499,0.938567,0.059874,0.933393,0.063091,334.381303,97.543575,140.882812,36066.0
3,1D-CNN,1738,0.5,0.929804,0.930950,0.924317,0.875000,0.934853,0.903937,0.977128,...,0.268933,0.035150,0.930521,0.074363,0.925872,0.075650,119.120289,8.362625,25.257812,6466.0
0,Logistic Regression,1738,0.5,0.892980,0.899155,0.886276,0.804843,0.920195,0.858663,0.960014,...,0.740990,0.085971,0.893790,0.143644,0.884455,0.155422,1.311484,0.102870,2.860352,277.0
1,Random Forest,1738,0.5,0.813003,0.814044,0.802041,0.702098,0.817590,0.755455,0.913979,...,0.352976,0.066379,0.812760,0.206345,0.791886,0.224068,140.182133,27.038050,3910.836914,48888.0


## LSTM Confusion Matrix, ROC, Precision-Recall, and Calibration

The model-quality confusion matrix uses the conventional 0.5 decision threshold. The second matrix uses MindWave's configured alert threshold of 0.85. The threshold sweep is descriptive: it does not select a threshold using the outer test labels.


In [7]:
lstm_oof = oof_df[oof_df["model"] == "LSTM"].sort_values("sample_index")
lstm_y = lstm_oof["y_true"].to_numpy()
lstm_p = lstm_oof["probability_stress"].to_numpy()

threshold_rows = []
for threshold in np.round(np.arange(0.05, 1.0, 0.05), 2):
    threshold_rows.append(metric_row(lstm_y, lstm_p, threshold=float(threshold)))
threshold_df = pd.DataFrame(threshold_rows)
save_table(threshold_df, "nested_loso_threshold_sweep.csv", "Threshold sensitivity computed only from corrected out-of-fold LSTM probabilities")

fraction_positive, mean_predicted = calibration_curve(lstm_y, lstm_p, n_bins=N_CALIBRATION_BINS, strategy="uniform")
calibration_df = pd.DataFrame({"mean_predicted_probability": mean_predicted, "observed_stress_frequency": fraction_positive})
save_table(calibration_df, "nested_loso_calibration_bins.csv", "Reliability-curve bins for corrected out-of-fold LSTM probabilities")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for axis, threshold in zip(axes, [0.5, PRODUCTION_THRESHOLD]):
    matrix = confusion_matrix(lstm_y, (lstm_p >= threshold).astype(int), labels=[0, 1])
    row_percent = matrix / matrix.sum(axis=1, keepdims=True)
    annotations = np.asarray([[f"{matrix[i, j]}\n{row_percent[i, j]:.1%}" for j in range(2)] for i in range(2)])
    sns.heatmap(matrix, annot=annotations, fmt="", cmap="Blues", cbar=False, ax=axis,
                xticklabels=["pred normal", "pred stress"], yticklabels=["true normal", "true stress"])
    axis.set_title(f"Threshold = {threshold:.2f}")
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
fig.tight_layout()
save_figure(fig, "nested_loso_confusion_matrices.png", "Corrected out-of-fold LSTM confusion matrices at 0.50 and 0.85")

fpr, tpr, _ = roc_curve(lstm_y, lstm_p)
precision, recall, _ = precision_recall_curve(lstm_y, lstm_p)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
axes[0].plot(fpr, tpr, color="#2a9d8f", label=f"AUC = {roc_auc_score(lstm_y, lstm_p):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="LOSO ROC curve")
axes[0].legend()
axes[1].plot(recall, precision, color="#e76f51", label=f"AP = {average_precision_score(lstm_y, lstm_p):.3f}")
axes[1].set(xlabel="Recall", ylabel="Precision", title="LOSO precision-recall curve")
axes[1].legend()
fig.tight_layout()
save_figure(fig, "nested_loso_roc_pr.png", "ROC and precision-recall curves from corrected out-of-fold LSTM predictions")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))
axes[0].plot(calibration_df["mean_predicted_probability"], calibration_df["observed_stress_frequency"], marker="o", color="#2a9d8f")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set(xlabel="Mean predicted stress probability", ylabel="Observed stress frequency", title="Probability reliability")
axes[1].plot(threshold_df["threshold"], threshold_df["stress_precision"], label="precision", color="#457b9d")
axes[1].plot(threshold_df["threshold"], threshold_df["stress_recall"], label="recall", color="#e76f51")
axes[1].plot(threshold_df["threshold"], threshold_df["stress_f1"], label="F1", color="#2a9d8f")
axes[1].axvline(PRODUCTION_THRESHOLD, linestyle="--", color="black", alpha=0.7, label="configured 0.85")
axes[1].set(xlabel="Threshold", ylabel="Score", title="Out-of-fold threshold sensitivity", ylim=(0, 1.02))
axes[1].legend()
fig.tight_layout()
save_figure(fig, "nested_loso_calibration_threshold.png", "Reliability diagram and threshold sensitivity for corrected LSTM probabilities")

pd.DataFrame([metric_row(lstm_y, lstm_p, threshold=0.5), metric_row(lstm_y, lstm_p, threshold=PRODUCTION_THRESHOLD)])


  -> wrote ml\evaluation\results\nested_loso_threshold_sweep.csv (19 rows)
  -> wrote ml\evaluation\results\nested_loso_calibration_bins.csv (10 rows)
  -> wrote ml\evaluation\plots\nested_loso_confusion_matrices.png
  -> wrote ml\evaluation\plots\nested_loso_roc_pr.png
  -> wrote ml\evaluation\plots\nested_loso_calibration_threshold.png


,n_windows,threshold,accuracy,balanced_accuracy,macro_f1,stress_precision,stress_recall,stress_f1,roc_auc,pr_auc,brier_score,log_loss,ece
0,1738,0.50,0.937860,0.934222,0.932301,0.904153,0.921824,0.912903,0.973173,0.944946,0.051183,0.21964,0.035499
1,1738,0.85,0.940736,0.928317,0.934221,0.942808,0.885993,0.913518,0.973173,0.944946,0.051183,0.21964,0.035499


## Subject-independent Sensor Ablation

Each sensor group is replaced with zero in normalized space, corresponding to mean-feature masking. The model and scaler still come from subjects other than the evaluated subject. This experiment measures model reliance on modalities; it does not reproduce every runtime fallback used by the Android application.


In [8]:
ablation_fold_rows = []
for (subject, modality), group in ablation_oof_df.groupby(["subject", "masked_modality"]):
    ablation_fold_rows.append({"subject": subject, "masked_modality": modality, **metric_row(group["y_true"].to_numpy(), group["probability_stress"].to_numpy())})
ablation_fold_df = pd.DataFrame(ablation_fold_rows)

ablation_summary_rows = []
for modality, group in ablation_oof_df.groupby("masked_modality"):
    metrics = metric_row(group["y_true"].to_numpy(), group["probability_stress"].to_numpy())
    per_fold = ablation_fold_df[ablation_fold_df["masked_modality"] == modality]
    ablation_summary_rows.append({
        "masked_modality": modality,
        **metrics,
        "subject_accuracy_mean": per_fold["accuracy"].mean(),
        "subject_accuracy_std": per_fold["accuracy"].std(ddof=0),
        "subject_macro_f1_mean": per_fold["macro_f1"].mean(),
        "subject_macro_f1_std": per_fold["macro_f1"].std(ddof=0),
    })
ablation_summary_df = pd.DataFrame(ablation_summary_rows)
baseline_accuracy = float(ablation_summary_df.loc[ablation_summary_df["masked_modality"] == "None", "accuracy"].iloc[0])
ablation_summary_df["accuracy_drop"] = baseline_accuracy - ablation_summary_df["accuracy"]

save_table(ablation_fold_df, "nested_loso_sensor_ablation_by_subject.csv", "Per-subject mean-masking ablation results using outer-fold LSTM models")
save_table(ablation_summary_df, "nested_loso_sensor_ablation.csv", "Aggregate subject-independent sensor-group ablation results")

plot_ablation = ablation_summary_df[ablation_summary_df["masked_modality"] != "None"].sort_values("accuracy_drop", ascending=False)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
sns.barplot(data=plot_ablation, x="masked_modality", y="accuracy_drop", color="#e76f51", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Masked sensor group")
ax.set_ylabel("Out-of-fold accuracy drop")
ax.set_title("Subject-independent modality ablation")
fig.tight_layout()
save_figure(fig, "nested_loso_sensor_ablation.png", "Out-of-fold LSTM accuracy drop after mean masking each sensor group")
ablation_summary_df.sort_values("accuracy_drop", ascending=False)


  -> wrote ml\evaluation\results\nested_loso_sensor_ablation_by_subject.csv (75 rows)
  -> wrote ml\evaluation\results\nested_loso_sensor_ablation.csv (5 rows)
  -> wrote ml\evaluation\plots\nested_loso_sensor_ablation.png


,masked_modality,n_windows,threshold,accuracy,balanced_accuracy,macro_f1,stress_precision,stress_recall,stress_f1,roc_auc,pr_auc,brier_score,log_loss,ece,subject_accuracy_mean,subject_accuracy_std,subject_macro_f1_mean,subject_macro_f1_std,accuracy_drop
0,EDA,1738,0.5,0.884350,0.871052,0.872835,0.843594,0.825733,0.834568,0.937572,0.898251,0.090190,0.374026,0.059904,0.884522,0.116902,0.867980,0.129566,0.053510
4,Temperature,1738,0.5,0.907365,0.906581,0.900156,0.844749,0.903909,0.873328,0.954074,0.893204,0.075841,0.345514,0.060520,0.907307,0.057856,0.900507,0.058981,0.030495
1,HRV/BVP,1738,0.5,0.915420,0.905419,0.907069,0.887231,0.871336,0.879211,0.951639,0.923810,0.071764,0.306989,0.050537,0.916072,0.095686,0.907084,0.100759,0.022440
2,Movement,1738,0.5,0.917722,0.908306,0.909666,0.889256,0.876221,0.882691,0.945473,0.919772,0.067036,0.317370,0.050189,0.918562,0.070629,0.909068,0.080016,0.020138
3,None,1738,0.5,0.937860,0.934222,0.932301,0.904153,0.921824,0.912903,0.973173,0.944946,0.051183,0.219640,0.035499,0.938567,0.059874,0.933393,0.063091,0.000000


## Simulated Personalization from Feedback

The outer-fold LSTM is adapted using a small, balanced set of labels from the held-out subject. Neighboring windows are purged from evaluation because 60-second windows begin every 15 seconds and therefore overlap. The before/after comparison is made on the same remaining windows.

This is an offline simulation of explicit feedback, not evidence from real application users.


In [9]:
if personalization_df.empty:
    display(Markdown("No subject had enough separated windows for the configured feedback budgets."))
else:
    personalization_summary_df = (
        personalization_df.groupby("feedback_budget", as_index=False)
        .agg(
            subjects=("subject", "nunique"),
            accuracy_before=("accuracy_before", "mean"),
            accuracy_after=("accuracy_after", "mean"),
            accuracy_delta_mean=("accuracy_delta", "mean"),
            accuracy_delta_std=("accuracy_delta", "std"),
            macro_f1_before=("macro_f1_before", "mean"),
            macro_f1_after=("macro_f1_after", "mean"),
            macro_f1_delta_mean=("macro_f1_delta", "mean"),
            macro_f1_delta_std=("macro_f1_delta", "std"),
            brier_before=("brier_before", "mean"),
            brier_after=("brier_after", "mean"),
        )
    )
    save_table(personalization_df, "personalization_by_subject.csv", "Per-subject performance before and after simulated feedback fine-tuning")
    save_table(personalization_summary_df, "personalization_summary.csv", "Average effect of different simulated feedback budgets")

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
    for axis, before_col, after_col, title in [
        (axes[0], "accuracy_before", "accuracy_after", "Accuracy"),
        (axes[1], "macro_f1_before", "macro_f1_after", "Macro-F1"),
    ]:
        long = personalization_df.melt(id_vars=["subject", "feedback_budget"], value_vars=[before_col, after_col], var_name="stage", value_name="score")
        long["stage"] = long["stage"].map({before_col: "before", after_col: "after"})
        sns.barplot(data=long, x="feedback_budget", y="score", hue="stage", errorbar="sd", ax=axis, palette=["#8ecae6", "#2a9d8f"])
        axis.set_ylim(0, 1)
        axis.set_title(f"Personalization {title}")
        axis.set_xlabel("Feedback examples")
    fig.tight_layout()
    save_figure(fig, "personalization_feedback_budgets.png", "Performance before and after simulated local personalization")
    display(personalization_summary_df)


  -> wrote ml\evaluation\results\personalization_by_subject.csv (45 rows)
  -> wrote ml\evaluation\results\personalization_summary.csv (3 rows)
  -> wrote ml\evaluation\plots\personalization_feedback_budgets.png


,feedback_budget,subjects,accuracy_before,accuracy_after,accuracy_delta_mean,accuracy_delta_std,macro_f1_before,macro_f1_after,macro_f1_delta_mean,macro_f1_delta_std,brier_before,brier_after
0,4,15,0.942580,0.945737,0.003157,0.009955,0.937411,0.940729,0.003318,0.010004,0.046750,0.045646
1,8,15,0.942274,0.946215,0.003941,0.012846,0.936676,0.940815,0.004139,0.012960,0.047126,0.046327
2,16,15,0.938709,0.942305,0.003596,0.012021,0.931320,0.935106,0.003786,0.012593,0.050382,0.049272


## Controlled Sample-weighted FedAvg Comparison

This experiment uses the same training subjects, held-out subjects, scaler, initialization, and fixed number of rounds for centralized and federated training. Each WESAD training subject acts as one client. Client parameters are weighted by the number of local windows.

The loop implements the FedAvg mathematics directly so that the comparison is independent of Flower/Ray runtime availability. `04_federated_learning.ipynb` remains the framework-level Flower experiment.


In [10]:
def weighted_average_weights(weight_sets: list[list[np.ndarray]], sample_counts: list[int]) -> list[np.ndarray]:
    total = float(sum(sample_counts))
    return [
        sum(weights[layer] * (count / total) for weights, count in zip(weight_sets, sample_counts)).astype(np.float32)
        for layer in range(len(weight_sets[0]))
    ]


def evaluate_keras_model(model: tf.keras.Model, values: np.ndarray, labels: np.ndarray) -> dict:
    probabilities = model.predict(values, verbose=0)[:, 1]
    return metric_row(labels, probabilities, threshold=0.5)


fl_test_subjects = outer_subjects[-1:] if not FULL_RUN else outer_subjects[-3:]
fl_train_subjects = [subject for subject in outer_subjects if subject not in fl_test_subjects]
fl_train_mask = np.isin(subjects_eval, fl_train_subjects)
fl_test_mask = np.isin(subjects_eval, fl_test_subjects)

fl_rows = []
fl_comm_rows = []
if len(fl_train_subjects) < 2 or not np.any(fl_test_mask):
    display(Markdown("Controlled FL comparison skipped because smoke mode does not contain enough subjects."))
else:
    fl_scaler = fit_scaler_3d(X_eval[fl_train_mask])
    fl_X_train = apply_scaler_3d(fl_scaler, X_eval[fl_train_mask])
    fl_y_train = y_eval[fl_train_mask]
    fl_X_test = apply_scaler_3d(fl_scaler, X_eval[fl_test_mask])
    fl_y_test = y_eval[fl_test_mask]

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    initial_model = build_lstm(input_shape=fl_X_train.shape[1:], n_classes=2, dropout=0.0)
    initial_weights = initial_model.get_weights()
    parameter_bytes = int(sum(weight.nbytes for weight in initial_weights))

    centralized = build_lstm(input_shape=fl_X_train.shape[1:], n_classes=2, dropout=0.0)
    centralized.set_weights(initial_weights)
    global_model = build_lstm(input_shape=fl_X_train.shape[1:], n_classes=2, dropout=0.0)
    global_model.set_weights(initial_weights)

    initial_metrics = evaluate_keras_model(global_model, fl_X_test, fl_y_test)
    fl_rows.append({"method": "Untrained initialization", "round": 0, **initial_metrics})

    for round_number in range(1, FL_ROUNDS + 1):
        centralized.fit(
            fl_X_train,
            fl_y_train,
            epochs=FL_LOCAL_EPOCHS,
            batch_size=64,
            class_weight=class_weights(fl_y_train),
            verbose=0,
        )
        fl_rows.append({"method": "Centralized", "round": round_number, **evaluate_keras_model(centralized, fl_X_test, fl_y_test)})

        local_weights = []
        local_counts = []
        base_weights = global_model.get_weights()
        for client_subject in fl_train_subjects:
            client_mask = subjects_eval == client_subject
            client_X = apply_scaler_3d(fl_scaler, X_eval[client_mask])
            client_y = y_eval[client_mask]
            client = build_lstm(input_shape=client_X.shape[1:], n_classes=2, dropout=0.0)
            client.set_weights(base_weights)
            client.fit(client_X, client_y, epochs=FL_LOCAL_EPOCHS, batch_size=32, class_weight=class_weights(client_y), verbose=0)
            local_weights.append(client.get_weights())
            local_counts.append(len(client_X))
            del client
        global_model.set_weights(weighted_average_weights(local_weights, local_counts))
        fl_rows.append({"method": "Sample-weighted FedAvg", "round": round_number, **evaluate_keras_model(global_model, fl_X_test, fl_y_test)})
        fl_comm_rows.append({
            "round": round_number,
            "clients": len(fl_train_subjects),
            "parameter_bytes_per_direction_per_client": parameter_bytes,
            "upload_bytes": parameter_bytes * len(fl_train_subjects),
            "download_bytes": parameter_bytes * len(fl_train_subjects),
            "total_bytes": 2 * parameter_bytes * len(fl_train_subjects),
        })
        gc.collect()

    fl_df = pd.DataFrame(fl_rows)
    fl_comm_df = pd.DataFrame(fl_comm_rows)
    save_table(fl_df, "controlled_fl_vs_centralized.csv", "Centralized and sample-weighted FedAvg results under a shared held-out-subject protocol")
    save_table(fl_comm_df, "controlled_fl_communication.csv", "Float32 parameter communication cost for the controlled FedAvg experiment")

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), sharex=True)
    for method, group in fl_df[fl_df["method"] != "Untrained initialization"].groupby("method"):
        axes[0].plot(group["round"], group["accuracy"], marker="o", label=method)
        axes[1].plot(group["round"], group["macro_f1"], marker="o", label=method)
    axes[0].set(title="Held-out-subject accuracy", xlabel="Round", ylabel="Accuracy", ylim=(0, 1.02))
    axes[1].set(title="Held-out-subject macro-F1", xlabel="Round", ylabel="Macro-F1", ylim=(0, 1.02))
    axes[0].legend()
    axes[1].legend()
    fig.tight_layout()
    save_figure(fig, "controlled_fl_vs_centralized.png", "Controlled centralized versus sample-weighted FedAvg performance")
    display(fl_df)


  -> wrote ml\evaluation\results\controlled_fl_vs_centralized.csv (7 rows)
  -> wrote ml\evaluation\results\controlled_fl_communication.csv (3 rows)
  -> wrote ml\evaluation\plots\controlled_fl_vs_centralized.png


,method,round,n_windows,threshold,accuracy,balanced_accuracy,macro_f1,stress_precision,stress_recall,stress_f1,roc_auc,pr_auc,brier_score,log_loss,ece
0,Untrained initialization,0,355,0.5,0.374648,0.347467,0.350096,0.202532,0.250000,0.223776,0.274711,0.256436,0.259435,0.712099,0.144910
1,Centralized,1,355,0.5,0.569014,0.659588,0.560291,0.454874,0.984375,0.622222,0.860270,0.808687,0.269693,0.754884,0.310582
2,Sample-weighted FedAvg,1,355,0.5,0.447887,0.483102,0.447848,0.348214,0.609375,0.443182,0.527223,0.438117,0.251879,0.696893,0.151301
3,Centralized,2,355,0.5,0.870423,0.886753,0.865637,0.756250,0.945312,0.840278,0.966238,0.953949,0.089871,0.283277,0.107812
4,Sample-weighted FedAvg,2,355,0.5,0.543662,0.610803,0.541388,0.432540,0.851562,0.573684,0.660277,0.555261,0.246585,0.686125,0.193594
5,Centralized,3,355,0.5,0.926761,0.927399,0.921610,0.875000,0.929688,0.901515,0.979213,0.969422,0.055758,0.183438,0.035133
6,Sample-weighted FedAvg,3,355,0.5,0.566197,0.645460,0.561433,0.450758,0.929688,0.607143,0.728593,0.621195,0.244619,0.681859,0.216219


## Artifact Manifest and Interpretation Boundaries

The manifest lists every table and plot generated by this run. Only artifacts generated in `full` mode are candidates for the thesis.

The notebook does **not** establish:

- physical-phone or physical-watch latency, CPU, memory, or battery consumption;
- Wear Data Layer latency and delivery reliability on a real paired device;
- usability, explanation comprehension, or trust among real users;
- production uptime or Kubernetes scaling behavior;
- privacy against attacks on model updates, secure aggregation, or differential privacy.

Those require device profiling, integration sessions, deployment evidence, or a user study rather than another offline notebook.


In [11]:
manifest_df = pd.DataFrame(generated).drop_duplicates(subset=["artifact"], keep="last")
manifest_path = RESULTS_DIR / "in_depth_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"Manifest -> {manifest_path.relative_to(REPO_ROOT)}")

headline = model_summary_df[[
    "model",
    "accuracy",
    "macro_f1",
    "stress_precision",
    "stress_recall",
    "roc_auc",
    "pr_auc",
    "brier_score",
    "ece",
]].copy()
display(Markdown("### Headline corrected subject-independent results"))
display(headline.round(4))
display(Markdown(f"Generated **{len(manifest_df)}** reusable artifacts in `{EVAL_ROOT.relative_to(REPO_ROOT)}`."))
manifest_df


Manifest -> ml\evaluation\results\in_depth_manifest.csv


### Headline corrected subject-independent results

,model,accuracy,macro_f1,stress_precision,stress_recall,roc_auc,pr_auc,brier_score,ece
2,MLP,0.9419,0.9369,0.9027,0.9365,0.9848,0.9755,0.0459,0.0267
4,LSTM,0.9379,0.9323,0.9042,0.9218,0.9732,0.9449,0.0512,0.0355
3,1D-CNN,0.9298,0.9243,0.8750,0.9349,0.9771,0.9529,0.0555,0.0351
0,Logistic Regression,0.8930,0.8863,0.8048,0.9202,0.9600,0.9248,0.0932,0.0860
1,Random Forest,0.8130,0.8020,0.7021,0.8176,0.9140,0.8553,0.1184,0.0664


Generated **23** reusable artifacts in `ml\evaluation`.

,artifact,kind,description
0,ml\evaluation\results\indepth_class_balance_by...,table,Class balance for subjects included in the in-...
1,ml\evaluation\results\indepth_environment.csv,table,"Software, dataset, and execution environment"
2,ml\evaluation\plots\indepth_class_balance_by_s...,plot,Class balance by evaluated WESAD subject
3,ml\evaluation\results\nested_loso_subject_spli...,table,Outer test and inner validation subjects for e...
4,ml\evaluation\results\nested_loso_fold_metrics...,table,Per-subject metrics for every model under corr...
5,ml\evaluation\results\nested_loso_oof_predicti...,table,Out-of-fold probabilities from untouched outer...
6,ml\evaluation\results\nested_loso_model_compar...,table,Comparable model-family results from concatena...
7,ml\evaluation\plots\nested_loso_model_comparis...,plot,Mean and subject-level variation for each eval...
8,ml\evaluation\plots\nested_loso_lstm_by_subjec...,plot,Corrected LSTM accuracy and macro-F1 by outer ...
9,ml\evaluation\results\nested_loso_threshold_sw...,table,Threshold sensitivity computed only from corre...
